<a href="https://colab.research.google.com/github/Sunidhishree/flyrank-ml-internship1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sunidhishree/flyrank-ml-internship1"
REPO_DIR = "flyrank-ml-internship1"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
os.makedirs("work/outputs", exist_ok=True)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
valid = df[df["avg_position"] > 0].copy()
print(df.shape)

(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in plain words: Every page gets checked against three conditions in priority order. First, is it stale (180+ days since update) and still getting meaningful traffic (impressions above the median)? If so, it's flagged for a refresh — this is the strongest, highest-priority signal, mirroring the session's refresh-flag logic. If not, is it ranking reasonably well (position 1-20) but underperforming on clicks (CTR below the median)? That's a metadata fix candidate — good position, weak appeal. If neither applies but the page is both declining and high-traffic, it goes to a monitoring bucket, since it's not yet clearly actionable but worth watching. Everything else gets no action.

Reason codes this rule can output:

STALE_HIGH_TRAFFIC → action: REFRESH
LOW_CTR_GOOD_POSITION → action: FIX_METADATA
DECLINING_TREND → action: MONITOR_CLOSELY
NONE → action: NO_ACTION

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
median_impressions = df["impressions_90d"].median()
median_ctr = valid["ctr"].median()

def score_row(row):
    stale = row["days_since_last_update"] >= 180
    high_traffic = row["impressions_90d"] >= median_impressions
    good_position = (row["avg_position"] > 0) and (row["avg_position"] <= 20)
    low_ctr = row["ctr"] < median_ctr
    declining = row["trend_direction"] == "down"

    if stale and high_traffic:
        return pd.Series({"reason_code": "STALE_HIGH_TRAFFIC", "action": "REFRESH", "score": row["impressions_90d"]})
    elif good_position and low_ctr:
        return pd.Series({"reason_code": "LOW_CTR_GOOD_POSITION", "action": "FIX_METADATA", "score": row["impressions_90d"]})
    elif declining and high_traffic:
        return pd.Series({"reason_code": "DECLINING_TREND", "action": "MONITOR_CLOSELY", "score": row["impressions_90d"] * 0.5})
    else:
        return pd.Series({"reason_code": "NONE", "action": "NO_ACTION", "score": 0})

rule_output = df.apply(score_row, axis=1)
queue = pd.concat([
    df[["content_id", "impressions_90d", "avg_position", "ctr", "days_since_last_update", "trend_direction"]],
    rule_output
], axis=1)
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows.")
print(queue["action"].value_counts())
queue.head(20)

Wrote 30000 rows.
action
NO_ACTION          13878
FIX_METADATA        8575
MONITOR_CLOSELY     7532
REFRESH               15
Name: count, dtype: int64


,content_id,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction,reason_code,action,score
0,content_36ff89c8214e,295097,7.3,0.05,104,stable,LOW_CTR_GOOD_POSITION,FIX_METADATA,295097.0
1,content_8451fc6f034d,272144,2.3,0.03,20,up,LOW_CTR_GOOD_POSITION,FIX_METADATA,272144.0
2,content_5fe46e04994d,517715,4.2,0.14,104,down,DECLINING_TREND,MONITOR_CLOSELY,258857.5
3,content_8c19996aa890,509252,2.5,0.15,20,down,DECLINING_TREND,MONITOR_CLOSELY,254626.0
4,content_4c36c775b818,463103,2.3,0.41,20,down,DECLINING_TREND,MONITOR_CLOSELY,231551.5
5,content_c84a0ab98e90,223271,7.8,0.03,20,stable,LOW_CTR_GOOD_POSITION,FIX_METADATA,223271.0
6,content_c8e9d6ab9013,208678,9.7,0.00,104,down,LOW_CTR_GOOD_POSITION,FIX_METADATA,208678.0
7,content_1a9e894be2e2,416180,4.0,0.23,22,down,DECLINING_TREND,MONITOR_CLOSELY,208090.0
8,content_2c2606c5d176,347399,4.2,0.53,104,down,DECLINING_TREND,MONITOR_CLOSELY,173699.5
9,content_91652435f57a,159590,7.8,0.06,104,stable,LOW_CTR_GOOD_POSITION,FIX_METADATA,159590.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

FIX_METADATA — content_36ff89c8214e: position 7.3, CTR 0.05%, 295,097 impressions. Confidence: high — strong position with unusually weak CTR is a clear metadata signal. Would be wrong if the low CTR reflects mismatched search intent rather than a fixable title/snippet issue.
FIX_METADATA — content_8451fc6f034d: position 2.3 (near top), CTR only 0.03%. Confidence: high — position this strong with CTR this low is unusual and worth investigating. Would be wrong if this page is a "zero-click" result type (e.g. answered directly in a featured snippet) where low CTR is structural, not a metadata failure.
MONITOR_CLOSELY — content_5fe46e04994d: 517,715 impressions, declining, position 4.2. Confidence: medium — highest-traffic page in the whole queue and declining, but position is still strong. Would be wrong if this is normal seasonal fluctuation rather than a real structural decline.
MONITOR_CLOSELY — content_8c19996aa890: position 2.5, declining. Confidence: medium — very strong position already; "declining" from an already-excellent spot may just be noise near a ceiling.
MONITOR_CLOSELY — content_4c36c775b818: CTR 0.41% (much higher than most rows here), declining. Confidence: low — this page's CTR is actually healthy relative to peers; the declining trend may not reflect a real problem worth escalating yet.
FIX_METADATA — content_c84a0ab98e90: position 7.8, CTR 0.03%. Confidence: high — same strong-position/weak-CTR pattern as row 1.
FIX_METADATA — content_c8e9d6ab9013: CTR literally 0.00%. Confidence: medium — a true 0.00% CTR with real impressions is either a genuine metadata failure or a possible tracking/data issue; worth a manual check before assuming it's purely a title problem.
MONITOR_CLOSELY — content_1a9e894be2e2: 416,180 impressions, declining, position 4.0. Confidence: medium — similar profile to row 3, another high-traffic page near the top declining.
MONITOR_CLOSELY — content_2c2606c5d176: CTR 0.53% (highest in this list), declining. Confidence: low — decent CTR undercuts the urgency here; likely a weaker pick than others in this bucket.
FIX_METADATA — content_91652435f57a: position 7.8, CTR 0.06%. Confidence: high — consistent with the pattern in rows 1 and 6.
MONITOR_CLOSELY — content_cb112fce36be: position 5.6, declining. Confidence: medium.
MONITOR_CLOSELY — content_9532f197bbc8: CTR 0.87% — by far the highest CTR in the top 20. Confidence: low — this page is performing well on clicks; flagging it as declining-and-urgent seems like the weakest call in the whole list.
FIX_METADATA — content_e12868d1f396: position 2.9, CTR 0.07%, only 7 days since update. Confidence: medium — recently updated already, so a metadata fix may have limited additional upside if the update didn't move CTR.
FIX_METADATA — content_97a86caf3a3d: position 6.4, CTR 0.07%. Confidence: high.
FIX_METADATA — content_453722754fea: CTR 0.01%, near-zero. Confidence: high — one of the clearest CTR problems in the list.
FIX_METADATA — content_c1fe78bc4e37: position 7.5, CTR 0.03%. Confidence: high.
FIX_METADATA — content_4a6607efcb46: trend is "up" but still flagged for CTR fix. Confidence: medium — worth noting this page is actually improving already; a metadata fix here is proactive, not urgent.
FIX_METADATA — content_4c76e9b13aea: also trending "up." Confidence: medium — same caveat as row 17.
FIX_METADATA — content_b115f7c74779: trending "up," CTR 0.03%. Confidence: medium — same caveat again.
FIX_METADATA — content_0919dd345d80: position 7.0, CTR 0.02%, only 7 days since update, declining. Confidence: high — strong signal despite being recently touched.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [3]:
# Leakage check: confirm no future-window or label-derived column was used as an INPUT to scoring
scoring_inputs = ["days_since_last_update", "impressions_90d", "avg_position", "ctr"]
print("Columns used in score_row logic:", scoring_inputs + ["trend_direction (used only as a gate, not scored on)"])
print("\ntrend_pct used anywhere in scoring?", "trend_pct" in scoring_inputs)
print("Any product/decision flags in dataset columns?",
      [c for c in df.columns if "flag" in c.lower() or "needs_" in c.lower()])

Columns used in score_row logic: ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'trend_direction (used only as a gate, not scored on)']

trend_pct used anywhere in scoring? False
Any product/decision flags in dataset columns? []


Leakage check: No future-window or label-derived columns were used as scoring inputs — trend_pct never appears in the scoring logic, and trend_direction is used only as a gate (declining vs. not), never assigned a score based on its own value. No product/decision-flag columns exist in this dataset.

Weak picks, honestly: Two real problems stand out in this top-20, not hypothetical ones. First, rows 12 and 9 (content_9532f197bbc8, CTR 0.87%; content_2c2606c5d176, CTR 0.53%) are labeled MONITOR_CLOSELY purely because they're declining and high-traffic — but their CTR is actually healthy relative to the rest of the queue, undercutting the case that anything is really wrong with them yet. Second, and more structurally: the action distribution is heavily skewed — only 15 pages out of 30,000 ever get REFRESH, while FIX_METADATA and MONITOR_CLOSELY dominate. Because score = impressions_90d in every branch, the "top 20" queue is really just "the 20 highest-traffic pages that happen to clear some gate," not a balanced view across all four action types. A stale, high-traffic page ranked #500 by raw impressions would never surface in this top-20 even if it's a stronger REFRESH candidate than most FIX_METADATA picks shown here. This is a real weakness in the scoring design worth fixing before Week 5 modeling — likely by normalizing/ranking within each action bucket rather than pooling everyone into one global sort.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.